### Need to first add the title to each of the records

In [3]:
import pandas as pd
import json
import numpy as np
from collections import defaultdict

print("Creating comprehensive document distribution dataset...")

# Step 1: Load and add patent titles + applicants
print("Loading data and adding patent titles and applicants...")
df = pd.read_parquet("../data/full_ds_spad-dataset-jsonl_sentence_clusters_junk_sentences_removed_semantic_chunked_w10_p95_embedded_chunks.parquet")

# Create lens_id to patent_title and applicants mapping
json_data_path = "../data/spad-dataset-jsonl.jsonl"
lens_id_to_title = {}
lens_id_to_applicants = {}

with open(json_data_path, 'r', encoding='utf-8') as file:
    for line_num, line in enumerate(file, 1):
        if line_num % 1000 == 0:
            print(f"Processing JSON line {line_num}...")
        try:
            patent_data = json.loads(line)
            lens_id = patent_data.get('lens_id')
            
            # Extract title
            title = None
            if 'biblio' in patent_data and 'invention_title' in patent_data['biblio']:
                invention_titles = patent_data['biblio']['invention_title']
                if invention_titles and len(invention_titles) > 0:
                    title = invention_titles[0].get('text', '')
            
            # Extract applicants
            applicants = []
            if 'biblio' in patent_data and 'parties' in patent_data['biblio'] and 'applicants' in patent_data['biblio']['parties']:
                applicants_data = patent_data['biblio']['parties']['applicants']
                for applicant in applicants_data:
                    if 'extracted_name' in applicant and 'value' in applicant['extracted_name']:
                        applicant_name = applicant['extracted_name']['value'].strip()
                        if applicant_name:
                            applicants.append(applicant_name)
            
            # Store the mappings
            if lens_id and title:
                lens_id_to_title[lens_id] = title
            if lens_id and applicants:
                lens_id_to_applicants[lens_id] = applicants
                
        except (json.JSONDecodeError, Exception) as e:
            continue

# Add title and applicants to dataframe
df['patent_title'] = df['lens_id'].map(lens_id_to_title)
df['applicants'] = df['lens_id'].map(lens_id_to_applicants)

print(f"Added titles for {df['patent_title'].notna().sum()} patents")
print(f"Added applicants for {df['applicants'].notna().sum()} patents")

# Debug: Show some applicant examples
print("\nSample applicants data:")
for i, (lens_id, applicants) in enumerate(lens_id_to_applicants.items()):
    if i < 5:  # Show first 5 examples
        print(f"  {lens_id}: {applicants}")

# Step 2: Get all unique topic IDs
print("Finding all unique topic IDs...")
all_topic_ids = set()
for col in ['abstract_paragraphs_topic_ids', 'claims_paragraphs_topic_ids', 'description_paragraphs_topic_ids']:
    for topic_array in df[col].dropna():
        all_topic_ids.update(list(topic_array))
all_topic_ids = sorted([tid for tid in all_topic_ids if tid != -1])
print(f"Found {len(all_topic_ids)} unique topic IDs")

# Step 3: Process topic counts and chunks
print("Processing topic assignments and chunks...")

def process_row_topics(row):
    topic_counts = defaultdict(int)
    topic_chunks = defaultdict(list)
    
    sections = [
        ('abstract', 'abstract_paragraphs_semantic_chunked_w10_p95', 'abstract_paragraphs_topic_ids'),
        ('description', 'description_paragraphs_semantic_chunked_w10_p95', 'description_paragraphs_topic_ids'),
        ('claims', 'claims_paragraphs_semantic_chunked_w10_p95', 'claims_paragraphs_topic_ids')
    ]
    
    for section_name, chunks_col, topic_ids_col in sections:
        chunks = row[chunks_col]
        topic_ids = row[topic_ids_col]
        
        # Convert numpy arrays to lists for consistent processing
        chunks_list = list(chunks)
        topic_ids_list = list(topic_ids)
        
        if len(chunks_list) == len(topic_ids_list):
            for chunk, topic_id in zip(chunks_list, topic_ids_list):
                if topic_id != -1:  # Skip invalid topic IDs
                    topic_counts[topic_id] += 1
                    topic_chunks[topic_id].append({
                        'text': str(chunk),
                        'section': section_name
                    })
    
    return topic_counts, topic_chunks

# Initialize data containers
topic_count_data = {}
topic_chunks_data = {}
for topic_id in all_topic_ids:
    topic_count_data[f'topic_{topic_id}_count'] = []
    topic_chunks_data[f'topic_{topic_id}_chunks'] = []

# Process all rows
for idx, row in df.iterrows():
    if idx % 100 == 0:
        print(f"Processing row {idx}/{len(df)}")
    
    topic_counts, topic_chunks = process_row_topics(row)
    
    # Add data for each topic
    for topic_id in all_topic_ids:
        topic_count_data[f'topic_{topic_id}_count'].append(topic_counts.get(topic_id, 0))
        topic_chunks_data[f'topic_{topic_id}_chunks'].append(topic_chunks.get(topic_id, []))

# Step 4: Create final document distribution DataFrame
print("Creating final document distribution DataFrame...")
essential_cols = ['lens_id', 'patent_title', 'applicants', 'jurisdiction', 'date_published']  # Added 'applicants'
document_distribution_df = df[essential_cols].copy()

# Add topic data using pd.concat for efficiency
count_df = pd.DataFrame(topic_count_data)
chunks_df = pd.DataFrame(topic_chunks_data)
document_distribution_df = pd.concat([document_distribution_df, count_df, chunks_df], axis=1)

print(f"Final document distribution DataFrame shape: {document_distribution_df.shape}")

# Step 5: Validation and summary
count_cols = [col for col in document_distribution_df.columns if col.endswith('_count')]
print(f"\nValidation - Topics with patent mentions:")
total_mentions = 0
for col in count_cols[:10]:  # Show first 10 topics
    non_zero_count = (document_distribution_df[col] > 0).sum()
    max_value = document_distribution_df[col].max()
    total_mentions += document_distribution_df[col].sum()
    print(f"  {col}: {non_zero_count} patents, max {max_value} mentions")

# NEW: Applicants validation
print(f"\nApplicants validation:")
patents_with_applicants = document_distribution_df['applicants'].notna().sum()
print(f"  📄 Patents with applicants: {patents_with_applicants}")
if patents_with_applicants > 0:
    sample_applicants = document_distribution_df[document_distribution_df['applicants'].notna()]['applicants'].iloc[0]
    print(f"  📝 Sample applicants: {sample_applicants}")
    
    # Count applicants per patent
    applicant_counts = document_distribution_df[document_distribution_df['applicants'].notna()]['applicants'].apply(len)
    print(f"  📊 Avg applicants per patent: {applicant_counts.mean():.1f}")
    print(f"  📊 Max applicants per patent: {applicant_counts.max()}")

print(f"\nSummary:")
print(f"  📄 Total patents: {len(document_distribution_df)}")
print(f"  🏷️ Total topics: {len(all_topic_ids)}")
print(f"  📊 Total topic mentions: {total_mentions}")
print(f"  👥 Patents with applicants: {patents_with_applicants}")
print(f"  💾 Ready to save as parquet file")

# Step 6: Save the document distribution file
output_path = "document_distribution.parquet"
document_distribution_df.to_parquet(output_path, index=False)
print(f"✅ Saved document distribution to: {output_path}")

# Show final sample
print(f"\nSample of final data:")
print(document_distribution_df[['lens_id', 'patent_title', 'applicants']].head(3))

Creating comprehensive document distribution dataset...
Loading data and adding patent titles and applicants...
Processing JSON line 1000...
Processing JSON line 2000...
Processing JSON line 3000...
Processing JSON line 4000...
Processing JSON line 5000...
Processing JSON line 6000...
Processing JSON line 7000...
Processing JSON line 8000...
Processing JSON line 9000...
Processing JSON line 10000...
Added titles for 951 patents
Added applicants for 951 patents

Sample applicants data:
  001-825-096-124-182: ['フラウンホーファー－ゲゼルシャフト・ツール・フェルデルング・デル・アンゲヴァンテン・フォルシュング・アインゲトラーゲネル・フェライン']
  000-462-269-867-675: ['PHILIPS NV']
  005-527-549-508-913: ['エヌアイ システムズ，インコーポレイテッド']
  003-338-140-301-073: ['FINDLAY EWAN', 'ST MICROELECTRONICS RES & DEV']
  004-267-921-266-208: ['PICOMETRIX LLC']
Finding all unique topic IDs...
Found 93 unique topic IDs
Processing topic assignments and chunks...
Processing row 0/951
Processing row 100/951
Processing row 200/951
Processing row 300/951
Processing row 400/951


In [4]:
document_distribution_df.iloc[0]

lens_id                                          006-022-986-985-120
patent_title       Compensated Photonic Device Structure And Fabr...
applicants                          [SIFOTONICS TECHNOLOGIES CO LTD]
jurisdiction                                                      US
date_published                                            2015-01-08
                                         ...                        
topic_88_chunks                                                   []
topic_89_chunks                                                   []
topic_90_chunks                                                   []
topic_91_chunks                                                   []
topic_92_chunks                                                   []
Name: 0, Length: 191, dtype: object